## Predictor Variable
<p align="justify">
Now that we have our water quality dataset, the next step is to gather the predictor variables from the <b>Landsat</b> and <b>TerraClimate</b> datasets. In this notebook, we demonstrate how to <b>load previously extracted satellite and climate data</b> from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.
</p>
<p align="justify">
For more detailed guidance on the original data extraction process, you can review the <a href="https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook">Landsat example notebook</a> and the <a href="https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook">TerraClimate example notebook</a> available on the Planetary Computer portal.
</p>

<p align="justify">We have used selected spectral bands — SWIR22 (Shortwave Infrared 2), NIR (Near Infrared), Green, and SWIR16 (Shortwave Infrared 1) — and computed key spectral indices such as NDMI (Normalized Difference Moisture Index) and MNDWI (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability. </p> <p align="justify"> In addition to Landsat features, we also incorporated the <b>Potential Evapotranspiration (PET)</b> variable from the <b>TerraClimate</b> dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters. </p> <ul> <li>SWIR22 – Sensitive to surface moisture and turbidity variations in water bodies.</li> <li>NIR – Helps in identifying vegetation and suspended matter in water.</li> <li>Green – Useful for detecting water color and surface reflectance changes.</li> <li>SWIR16 – Provides information on surface dryness and sediment concentration.</li> <li>NDMI – Derived from NIR and SWIR16, indicates moisture and vegetation-water interaction.</li> <li>MNDWI – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.</li> <li>PET – Extracted from the TerraClimate dataset, represents the potential evapotranspiration that influences hydrological and water quality dynamics.</li> </ul>

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [7]:
import pandas as pd
Water_Quality_df = pd.read_csv('./data/original/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [8]:
landsat_train_features = pd.read_csv('./data/original/landsat_features_training.csv')
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [9]:
Terraclimate_df = pd.read_csv('./data/original/terraclimate_features_training.csv')
Terraclimate_df.head()

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


In [10]:
from utils.pipeline import *
MERGE_KEYS = ['Latitude', 'Longitude', 'Sample Date']
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, keys=MERGE_KEYS)
wq_data.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


## Preprocess data

In [13]:
wq_data = wq_data[['swir22','NDMI','MNDWI','pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]
wq_data

,swir22,NDMI,MNDWI,pet,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,7645.0,0.185538,0.195595,174.20000,128.912,555.0,10.0
1,10574.0,0.124566,-0.180134,124.10000,74.720,162.9,163.0
2,14201.0,-0.083293,-0.252805,127.50000,89.254,573.0,80.0
3,11403.0,0.048048,-0.105416,129.70000,82.000,203.6,101.0
4,9643.0,0.141147,-0.142683,129.20000,56.100,145.1,151.0
...,...,...,...,...,...,...,...
9314,14443.0,-0.034236,-0.239858,166.30000,38.900,134.0,20.0
9315,14710.0,-0.042921,-0.246928,182.40001,115.800,388.0,20.0
9316,16281.0,-0.100999,-0.260754,207.80000,104.874,835.0,148.0
9317,15724.5,-0.111396,-0.250042,222.80000,128.000,305.0,28.0


## Run pipeline


In [23]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")


Training Model for Total Alkalinity


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Improve score (reduce overfitting)

Options that usually help:
1. **Regularized XGBoost** – use `DEFAULT_REGULARIZED_XGB` (shallower trees, L1/L2, subsample).
2. **No PCA** – with only 4–7 features, PCA often keeps almost all dimensions; `use_pca=False` can generalize better.
3. **All 7 features** – keep `nir`, `green`, `swir16` in the feature list above (don’t subset to 4).
4. **Hyperparameter tuning** – `run_pipeline_with_tuning()` (slower, uses GridSearchCV).

In [21]:
# Option A: same pipeline with stronger regularization (no tuning)
# For more signal, use all 7 features: in the preprocessing cell use
# wq_data = wq_data[['nir','green','swir16','swir22','NDMI','MNDWI','pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]
from utils.pipeline import run_pipeline, DEFAULT_REGULARIZED_XGB

model_TA_r, results_TA_r = run_pipeline(X, y_TA, "Total Alkalinity", use_pca=False, xgb_params=DEFAULT_REGULARIZED_XGB)
model_EC_r, results_EC_r = run_pipeline(X, y_EC, "Electrical Conductance", use_pca=False, xgb_params=DEFAULT_REGULARIZED_XGB)
model_DRP_r, results_DRP_r = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus", use_pca=False, xgb_params=DEFAULT_REGULARIZED_XGB)

# Option B (slower): GridSearchCV over max_depth, learning_rate, reg_alpha, reg_lambda
# from utils.pipeline import run_pipeline_with_tuning
# model_TA_t, results_TA_t = run_pipeline_with_tuning(X, y_TA, "Total Alkalinity", cv=3, use_pca=False)


Training Model for Total Alkalinity

Train Evaluation:
R²: 0.386
RMSE: 58.243

Test Evaluation:
R²: 0.321
RMSE: 62.171

Training Model for Electrical Conductance

Train Evaluation:
R²: 0.427
RMSE: 258.977

Test Evaluation:
R²: 0.361
RMSE: 273.193

Training Model for Dissolved Reactive Phosphorus

Train Evaluation:
R²: 0.372
RMSE: 40.298

Test Evaluation:
R²: 0.292
RMSE: 43.151
